# Step 4 — Teacher network for knowledge distillation

**What this step does:** trains a much larger network with Step 2's recipe. With 36.5M parameters it is **not** a
candidate for the final model; Step 5 uses it as a teacher. Its validation accuracy shows how much better than the
student it is.

- Network: WRN-28-10 built from the same pre-activation blocks, channels 160/320/640, blocks 4/4/4, stem 16,
  **36,479,194 parameters**
- Training: Step 2's recipe, **200 epochs**; about 5–7× the time of Step 2
- Depends on: nothing — start it early, on the fastest GPU, because Step 5 waits for it
- Output: `runs/teacher_wrn28_10/` (`last.pt` ≈ 440 MB is read by Step 5); no submission file

**How to run:** Kernel → Restart Kernel and Run All Cells. If the run stops (session ended, kernel restarted), run all
cells again: training continues from `runs/teacher_wrn28_10/last.pt`, and a finished run is not trained again.
To start this step from scratch, delete `runs/teacher_wrn28_10/` first.


## 1. Imports and settings

In [1]:
import copy
import csv
import glob
import json
import math
import os
import platform
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

RAW_DIR = "data/raw"        # extracted competition data (train_images/, test_images/, train_labels.csv, test.csv,
                            # sample_submission.csv); they may also sit one folder deeper
CACHE_DIR = "data/cache"    # numpy cache built in Section 2, shared by all step notebooks
RUNS_DIR = "runs"           # one sub-folder per run


def _available_cpus():
    try:
        return len(os.sched_getaffinity(0))
    except AttributeError:
        return os.cpu_count() or 2


# DataLoader worker processes. On Windows and macOS, workers are started with "spawn" and cannot load classes
# defined inside a notebook, so data loading runs in the main process there.
NUM_WORKERS = 0 if platform.system() in ("Windows", "Darwin") else max(2, min(10, _available_cpus() - 2))
print("DataLoader workers:", NUM_WORKERS)

torch 2.8.0+cu128 | CUDA available: True
DataLoader workers: 10


## 2. Configuration of this step

In [2]:
CONFIG = {
    "name": "teacher_wrn28_10",
    "seed": 42,
    "val_size": 5000,               # stratified validation split from the 50k training images (500 per class)
    "full_train": False,            # True = train on all 50k images without a validation split (Step 6)
    "model": {"widths": [160, 320, 640], "blocks": [4, 4, 4], "stem_channels": 16},   # WRN-28-10, 36,479,194 parameters
    "param_limit": None,            # the teacher is not a submission candidate, so no limit
    # optimisation (same in every step)
    "epochs": 200,
    "batch_size": 128,
    "lr": 0.1,                      # peak learning rate of SGD with Nesterov momentum
    "momentum": 0.9,
    "weight_decay": 5e-4,           # on conv / linear weights only
    "warmup_epochs": 5,             # linear warm-up, then cosine decay to 0
    # added in Step 2
    "trivialaugment": True,         # TrivialAugmentWide after random crop + horizontal flip
    "cutout": 16,                   # Cutout square size in pixels
    "label_smoothing": 0.1,
    "ema_decay": 0.999,             # moving average of the weights, used for evaluation and prediction
}

## 3. Data preparation

Builds the numpy cache `data/cache/` from the competition files on the first run (shared by all step notebooks); later runs skip this.

In [3]:
def read_csv_rows(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f))


def find_raw_root(raw_dir):
    # the folder that contains train_labels.csv (directly in raw_dir or one or more levels below)
    hits = glob.glob(os.path.join(raw_dir, "**", "train_labels.csv"), recursive=True)
    if not hits:
        raise FileNotFoundError(f"train_labels.csv not found under {raw_dir}: extract the competition data there")
    return os.path.dirname(min(hits, key=lambda p: p.count(os.sep)))


def load_images(folder, filenames):
    arr = np.empty((len(filenames), 32, 32, 3), dtype=np.uint8)
    for i, fn in enumerate(filenames):
        img = Image.open(os.path.join(folder, fn)).convert("RGB")
        assert img.size == (32, 32), f"{fn}: unexpected size {img.size}"
        arr[i] = np.asarray(img)
    return arr


def _atomic_write(path, write_fn):
    tmp = f"{path}.{os.getpid()}.tmp"
    with open(tmp, "wb") as f:
        write_fn(f)
    os.replace(tmp, path)


def prepare_data(raw_dir, out_dir):
    root = find_raw_root(raw_dir)
    print("reading competition files from", root)
    os.makedirs(out_dir, exist_ok=True)

    # training set
    rows = sorted(read_csv_rows(os.path.join(root, "train_labels.csv")), key=lambda r: int(r["id"]))
    train_x = load_images(os.path.join(root, "train_images"), [r["filename"] for r in rows])
    train_y = np.array([int(r["label"]) for r in rows], dtype=np.int64)
    assert train_y.min() >= 0 and train_y.max() <= 9

    # test set, ordered exactly like sample_submission.csv
    test_rows = {int(r["id"]): r["filename"] for r in read_csv_rows(os.path.join(root, "test.csv"))}
    sub_ids = [int(r["id"]) for r in read_csv_rows(os.path.join(root, "sample_submission.csv"))]
    assert set(sub_ids) == set(test_rows), "test.csv ids do not match sample_submission.csv ids"
    test_x = load_images(os.path.join(root, "test_images"), [test_rows[i] for i in sub_ids])
    test_ids = np.array(sub_ids, dtype=np.int64)

    # normalisation statistics from the training images only
    xf = train_x.astype(np.float64) / 255.0
    mean, std = xf.mean(axis=(0, 1, 2)).tolist(), xf.std(axis=(0, 1, 2)).tolist()

    # written atomically and meta.json last, so notebooks started at the same time never read a half-written cache
    for name, arr in [("train_x", train_x), ("train_y", train_y), ("test_x", test_x), ("test_ids", test_ids)]:
        _atomic_write(os.path.join(out_dir, f"{name}.npy"), lambda f, a=arr: np.save(f, a))
    _atomic_write(os.path.join(out_dir, "meta.json"),
                  lambda f: f.write(json.dumps({"mean": mean, "std": std}, indent=2).encode()))

    print(f"train: {train_x.shape}, class counts {np.bincount(train_y).tolist()}")
    print(f"test : {test_x.shape}")
    print(f"mean {np.round(mean, 4).tolist()}  std {np.round(std, 4).tolist()}")


if os.path.exists(os.path.join(CACHE_DIR, "meta.json")):
    print(f"data cache found in {CACHE_DIR}")
else:
    prepare_data(RAW_DIR, CACHE_DIR)

data cache found in data/cache


## 4. Dataset, train / validation split and augmentation

In [4]:
def load_cache(cache_dir):
    x = np.load(os.path.join(cache_dir, "train_x.npy"))
    y = np.load(os.path.join(cache_dir, "train_y.npy"))
    test_x = np.load(os.path.join(cache_dir, "test_x.npy"))
    test_ids = np.load(os.path.join(cache_dir, "test_ids.npy"))
    with open(os.path.join(cache_dir, "meta.json")) as f:
        meta = json.load(f)
    return x, y, test_x, test_ids, meta


def stratified_split(y, val_size, seed):
    # (train_idx, val_idx) with val_size images, the same number per class; identical in every run
    rng = np.random.RandomState(seed)
    classes = np.unique(y)
    per_class = val_size // len(classes)
    val_idx = np.sort(np.concatenate(
        [rng.choice(np.where(y == c)[0], per_class, replace=False) for c in classes]))
    train_idx = np.setdiff1d(np.arange(len(y)), val_idx)
    return train_idx, val_idx


class ArrayDataset(Dataset):
    def __init__(self, x, y, transform):
        self.x, self.y, self.transform = x, y, transform

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        img = self.transform(Image.fromarray(self.x[i]))
        label = -1 if self.y is None else int(self.y[i])
        return img, label


class Cutout:
    # added in Step 2: zero out one random size x size square of the normalized image (DeVries & Taylor, 2017)
    def __init__(self, size):
        self.size = size

    def __call__(self, img):
        _, h, w = img.shape
        cy, cx = int(torch.randint(h, (1,))), int(torch.randint(w, (1,)))
        y1, y2 = max(0, cy - self.size // 2), min(h, cy + self.size // 2)
        x1, x2 = max(0, cx - self.size // 2), min(w, cx + self.size // 2)
        img = img.clone()
        img[:, y1:y2, x1:x2] = 0.0
        return img


def build_transforms(cfg, mean, std):
    train = [T.RandomCrop(32, padding=4, padding_mode="reflect"), T.RandomHorizontalFlip()]
    if cfg["trivialaugment"]:                       # added in Step 2
        train.append(T.TrivialAugmentWide())
    train += [T.ToTensor(), T.Normalize(mean, std)]
    if cfg["cutout"] > 0:                           # added in Step 2
        train.append(Cutout(cfg["cutout"]))
    test = [T.ToTensor(), T.Normalize(mean, std)]
    return T.Compose(train), T.Compose(test)

## 5. Model: pre-activation ResNet

Stem 3×3 conv (no max-pool) → 3 stages of pre-activation basic blocks (the first block of stages 2 and 3 has stride 2) → BN-ReLU → global average pooling → FC. Residual block: $y = S(x) + F(x)$ with $F$ = BN-ReLU-Conv3×3-BN-ReLU-Conv3×3 and $S$ = identity, or AvgPool(2) + Conv1×1 when the block changes the resolution or width.

In [5]:
class PreActBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_ch != out_ch:
            layers = [nn.AvgPool2d(stride, stride)] if stride != 1 else []
            layers.append(nn.Conv2d(in_ch, out_ch, 1, 1, 0, bias=False))
            self.shortcut = nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        # the projection shortcut takes the pre-activated input (as in PreAct-ResNet / WRN)
        sc = x if self.shortcut is None else self.shortcut(out)
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + sc


class PreActResNet(nn.Module):
    def __init__(self, widths=(64, 128, 256), blocks=(4, 4, 3), stem_channels=64, num_classes=10):
        super().__init__()
        assert len(widths) == len(blocks)
        self.stem = nn.Conv2d(3, stem_channels, 3, 1, 1, bias=False)
        layers, in_ch = [], stem_channels
        for stage, (out_ch, n) in enumerate(zip(widths, blocks)):
            for i in range(n):
                stride = 2 if (stage > 0 and i == 0) else 1
                layers.append(PreActBlock(in_ch, out_ch, stride))
                in_ch = out_ch
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_ch)
        self.fc = nn.Linear(in_ch, num_classes)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.zeros_(m.bias)

    def forward(self, x):
        out = self.layers(self.stem(x))
        out = F.relu(self.bn(out))
        out = F.adaptive_avg_pool2d(out, 1).flatten(1)
        return self.fc(out)


def build_model(mcfg):
    return PreActResNet(widths=tuple(mcfg["widths"]), blocks=tuple(mcfg["blocks"]),
                        stem_channels=mcfg["stem_channels"])


def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print(f"{CONFIG['name']}: {count_trainable_params(build_model(CONFIG['model'])):,} trainable parameters")

teacher_wrn28_10: 36,479,194 trainable parameters


## 6. Training utilities

In [6]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


class CSVLogger:
    def __init__(self, path, fields, append=False):
        self.path, self.fields = path, fields
        if not (append and os.path.exists(path)):
            with open(path, "w", newline="") as f:
                csv.DictWriter(f, fields).writeheader()

    def log(self, row):
        with open(self.path, "a", newline="") as f:
            csv.DictWriter(f, self.fields).writerow(row)


def build_optimizer(model, cfg):
    # SGD with Nesterov momentum; weight decay on conv / linear weights only (not BatchNorm parameters or biases)
    decay, no_decay = [], []
    for p in model.parameters():
        if p.requires_grad:
            (no_decay if p.ndim <= 1 else decay).append(p)
    groups = [{"params": decay, "weight_decay": cfg["weight_decay"]},
              {"params": no_decay, "weight_decay": 0.0}]
    return torch.optim.SGD(groups, lr=cfg["lr"], momentum=cfg["momentum"], nesterov=True)


def warmup_cosine(total_steps, warmup_steps):
    # per-step learning-rate multiplier: linear warm-up, then cosine decay to 0
    def fn(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))
    return fn


class EMA:
    # added in Step 2: exponential moving average of the weights (BatchNorm buffers included)
    def __init__(self, model, decay):
        self.decay, self.updates = decay, 0
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        self.updates += 1
        d = min(self.decay, (1 + self.updates) / (10 + self.updates))   # smaller decay during the first steps
        msd = model.state_dict()
        for k, v in self.module.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(msd[k].detach(), alpha=1.0 - d)
            else:
                v.copy_(msd[k])


@torch.no_grad()
def predict_probs(model, loader, device, tta_flip=False):
    model.eval()
    probs, labels = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True).to(memory_format=torch.channels_last)
        p = F.softmax(model(x).float(), dim=1)
        if tta_flip:                                # added in Step 2: average with the horizontally flipped image
            p = 0.5 * (p + F.softmax(model(torch.flip(x, dims=[3])).float(), dim=1))
        probs.append(p.cpu())
        labels.append(y)
    return torch.cat(probs), torch.cat(labels)


def accuracy(model, loader, device, tta_flip=False):
    probs, labels = predict_probs(model, loader, device, tta_flip)
    return (probs.argmax(1) == labels).float().mean().item()

## 7. Training and prediction functions

`train(cfg, resume)` writes `runs/<name>/`: `config.json`, `run_info.json` (GPU, parameter count), `log.csv` (one row per epoch), `last.pt` (final epoch) and `best.pt` (best validation epoch). `predict(ckpt, val, out)` writes `eval.json` (validation accuracy) and, with `out`, the submission file.

In [7]:
def train(cfg, resume=False):
    run_dir = os.path.join(RUNS_DIR, cfg["name"])
    os.makedirs(run_dir, exist_ok=True)
    with open(os.path.join(run_dir, "config.json"), "w") as f:
        json.dump(cfg, f, indent=2)

    set_seed(cfg["seed"])
    device = get_device()
    torch.backends.cudnn.benchmark = True
    device_name = torch.cuda.get_device_name(0) if device.type == "cuda" else str(device)

    # ---------------- data
    x, y, _, _, meta = load_cache(CACHE_DIR)
    train_tf, test_tf = build_transforms(cfg, meta["mean"], meta["std"])
    if cfg["full_train"]:
        train_idx, val_idx = np.arange(len(y)), None
    else:
        train_idx, val_idx = stratified_split(y, cfg["val_size"], cfg["seed"])
    loader_kw = dict(num_workers=NUM_WORKERS, pin_memory=device.type == "cuda",
                     persistent_workers=NUM_WORKERS > 0)
    train_loader = DataLoader(ArrayDataset(x[train_idx], y[train_idx], train_tf),
                              batch_size=cfg["batch_size"], shuffle=True, drop_last=True, **loader_kw)
    val_loader = None
    if val_idx is not None:
        val_loader = DataLoader(ArrayDataset(x[val_idx], y[val_idx], test_tf),
                                batch_size=512, shuffle=False, **loader_kw)

    # ---------------- model
    model = build_model(cfg["model"]).to(device).to(memory_format=torch.channels_last)
    n_params = count_trainable_params(model)
    print(f"[{cfg['name']}] device: {device_name} | trainable parameters: {n_params:,} | "
          f"training images: {len(train_idx)} | epochs: {cfg['epochs']}")
    if cfg["param_limit"] is not None:
        assert n_params <= cfg["param_limit"], f"{n_params:,} trainable parameters > limit {cfg['param_limit']:,}"
    ema = EMA(model, cfg["ema_decay"])

    # ---------------- optimisation
    opt = build_optimizer(model, cfg)
    steps_per_epoch = len(train_loader)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, warmup_cosine(cfg["epochs"] * steps_per_epoch, cfg["warmup_epochs"] * steps_per_epoch))
    use_amp = device.type == "cuda"                     # mixed precision on GPU
    amp_dtype = torch.bfloat16 if use_amp and torch.cuda.is_bf16_supported() else torch.float16
    # torch >= 2.3: torch.amp.GradScaler("cuda"); older versions: torch.cuda.amp.GradScaler
    _GradScaler = (lambda **kw: torch.amp.GradScaler("cuda", **kw)) if hasattr(torch.amp, "GradScaler") else torch.cuda.amp.GradScaler
    scaler = _GradScaler(enabled=use_amp and amp_dtype == torch.float16)

    # ---------------- resume from runs/<name>/last.pt
    start_epoch, best_acc = 0, 0.0
    last_path, best_path = os.path.join(run_dir, "last.pt"), os.path.join(run_dir, "best.pt")
    if resume and os.path.exists(last_path):
        ck = torch.load(last_path, map_location="cpu", weights_only=False)
        model.load_state_dict(ck["model"])
        ema.module.load_state_dict(ck["ema"])
        ema.updates = ck["ema_updates"]
        opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"])
        scaler.load_state_dict(ck["scaler"])
        start_epoch, best_acc = ck["epoch"] + 1, ck["best_acc"]
        print(f"resuming from epoch {start_epoch}" if start_epoch < cfg["epochs"]
              else f"run already finished ({start_epoch} epochs)")

    with open(os.path.join(run_dir, "run_info.json"), "w") as f:
        json.dump({"device": device_name, "torch": torch.__version__, "n_params": n_params,
                   "start_epoch": start_epoch, "epochs": cfg["epochs"]}, f, indent=2)
    fields = ["epoch", "lr", "train_loss", "train_acc", "val_acc", "val_acc_ema", "time_s"]
    logger = CSVLogger(os.path.join(run_dir, "log.csv"), fields, append=resume)

    # ---------------- training loop
    ls = cfg["label_smoothing"]
    for epoch in range(start_epoch, cfg["epochs"]):
        model.train()
        t0, loss_sum, correct, seen = time.time(), 0.0, 0, 0
        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True).to(memory_format=torch.channels_last)
            yb = yb.to(device, non_blocking=True)

            with torch.autocast(device.type, dtype=amp_dtype, enabled=use_amp):
                logits = model(xb)
                loss = F.cross_entropy(logits, yb, label_smoothing=ls)

            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            ema.update(model)

            loss_sum += loss.item() * xb.size(0)
            correct += (logits.argmax(1) == yb).sum().item()
            seen += xb.size(0)

        row = {"epoch": epoch + 1, "lr": f"{opt.param_groups[0]['lr']:.5f}",
               "train_loss": f"{loss_sum / seen:.4f}", "train_acc": f"{correct / seen:.4f}",
               "val_acc": "", "val_acc_ema": "", "time_s": f"{time.time() - t0:.1f}"}
        score = None
        if val_loader is not None:
            row["val_acc"] = f"{accuracy(model, val_loader, device):.4f}"
            row["val_acc_ema"] = f"{accuracy(ema.module, val_loader, device):.4f}"
            score = float(row["val_acc_ema"])          # best.pt is chosen with the EMA weights
        logger.log(row)
        print("  ".join(f"{k}={v}" for k, v in row.items() if v != ""), flush=True)

        ck = {"model": model.state_dict(),
              "ema": ema.module.state_dict(), "ema_updates": ema.updates,
              "opt": opt.state_dict(), "sched": sched.state_dict(), "scaler": scaler.state_dict(),
              "epoch": epoch, "best_acc": best_acc, "cfg": cfg, "n_params": n_params}
        if score is not None and score > best_acc:
            best_acc = ck["best_acc"] = score
            torch.save(ck, best_path)
        torch.save(ck, last_path)

    if val_loader is not None:
        print(f"best validation accuracy: {best_acc:.4f}")
    print(f"done. run folder: {run_dir}")
    return run_dir


def predict(ckpt, val=False, out=None):
    # evaluate a checkpoint on the validation split and / or write test predictions (sample_submission.csv order)
    device = get_device()
    ck = torch.load(ckpt, map_location="cpu", weights_only=False)
    cfg = ck["cfg"]
    weights = "ema"                                     # added in Step 2: predict with the EMA weights
    model = build_model(cfg["model"])
    model.load_state_dict(ck[weights])
    n_params = count_trainable_params(model)
    print(f"{ckpt}: {weights} weights, epoch {ck['epoch'] + 1}, {n_params:,} trainable parameters")
    model = model.to(device).to(memory_format=torch.channels_last).eval()
    results = {"checkpoint": ckpt, "weights": weights, "epoch": ck["epoch"] + 1, "n_params": n_params}

    x, y, test_x, test_ids, meta = load_cache(CACHE_DIR)
    _, test_tf = build_transforms(cfg, meta["mean"], meta["std"])

    if val:
        if cfg["full_train"]:
            print("trained on all 50k images: no validation split")
        else:
            _, val_idx = stratified_split(y, cfg["val_size"], cfg["seed"])
            vl = DataLoader(ArrayDataset(x[val_idx], y[val_idx], test_tf), batch_size=512)
            results["val_acc"] = accuracy(model, vl, device)
            results["val_acc_flip_tta"] = accuracy(model, vl, device, tta_flip=True)
            print(f"val acc (no TTA):   {results['val_acc']:.4f}")
            print(f"val acc (flip TTA): {results['val_acc_flip_tta']:.4f}")

    if out is not None:
        tl = DataLoader(ArrayDataset(test_x, None, test_tf), batch_size=512)
        probs, _ = predict_probs(model, tl, device, tta_flip=True)
        preds = probs.argmax(1).tolist()
        assert len(preds) == len(test_ids)
        with open(out, "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["id", "label"])
            for i, p in zip(test_ids.tolist(), preds):
                w.writerow([i, int(p)])
        results["submission"] = out
        print(f"wrote {len(preds)} predictions to {out}; "
              f"predicted class counts {torch.bincount(torch.tensor(preds), minlength=10).tolist()}")

    with open(os.path.join(os.path.dirname(ckpt), "eval.json"), "w") as f:
        json.dump(results, f, indent=2)
    return results


def print_log(run_dir, every=20):
    # compact training record (every `every`-th epoch and the last one), kept in the saved notebook
    with open(os.path.join(run_dir, "log.csv")) as f:
        rows = list(csv.DictReader(f))
    for r in rows:
        if int(r["epoch"]) % every == 0 or r is rows[-1]:
            print("  ".join(f"{k}={v}" for k, v in r.items() if v))

## 8. Run

In [8]:
run_dir = train(CONFIG, resume=True)
predict(os.path.join(run_dir, "last.pt"), val=True)
print_log(run_dir)

[teacher_wrn28_10] device: NVIDIA A100-SXM4-40GB | trainable parameters: 36,479,194 | training images: 45000 | epochs: 200


epoch=1  lr=0.02006  train_loss=2.0673  train_acc=0.2587  val_acc=0.4174  val_acc_ema=0.5146  time_s=29.2


epoch=2  lr=0.04006  train_loss=1.7849  train_acc=0.4088  val_acc=0.5772  val_acc_ema=0.6736  time_s=12.5


epoch=3  lr=0.06006  train_loss=1.6340  train_acc=0.4870  val_acc=0.5630  val_acc_ema=0.7444  time_s=12.5


epoch=4  lr=0.08006  train_loss=1.5359  train_acc=0.5350  val_acc=0.6422  val_acc_ema=0.7872  time_s=12.5


epoch=5  lr=0.10000  train_loss=1.4760  train_acc=0.5648  val_acc=0.7362  val_acc_ema=0.8072  time_s=12.6


... (194 outputs left out)

epoch=200  lr=0.00000  train_loss=0.6606  train_acc=0.9275  val_acc=0.9760  val_acc_ema=0.9762  time_s=12.5


best validation accuracy: 0.9764
done. run folder: runs/teacher_wrn28_10


runs/teacher_wrn28_10/last.pt: ema weights, epoch 200, 36,479,194 trainable parameters


val acc (no TTA):   0.9762
val acc (flip TTA): 0.9792
epoch=20  lr=0.09855  train_loss=1.1720  train_acc=0.7049  val_acc=0.8534  val_acc_ema=0.9080  time_s=12.5
epoch=40  lr=0.09226  train_loss=1.0904  train_acc=0.7397  val_acc=0.8820  val_acc_ema=0.9304  time_s=12.4
epoch=60  lr=0.08162  train_loss=1.0459  train_acc=0.7564  val_acc=0.8854  val_acc_ema=0.9396  time_s=12.5
epoch=80  lr=0.06773  train_loss=1.0016  train_acc=0.7755  val_acc=0.9002  val_acc_ema=0.9462  time_s=12.4
epoch=100  lr=0.05201  train_loss=0.9563  train_acc=0.7993  val_acc=0.9118  val_acc_ema=0.9522  time_s=12.5
epoch=120  lr=0.03609  train_loss=0.9038  train_acc=0.8218  val_acc=0.9334  val_acc_ema=0.9580  time_s=12.5
epoch=140  lr=0.02160  train_loss=0.8361  train_acc=0.8503  val_acc=0.9508  val_acc_ema=0.9650  time_s=12.6
epoch=160  lr=0.01003  train_loss=0.7715  train_acc=0.8797  val_acc=0.9570  val_acc_ema=0.9698  time_s=12.5
epoch=180  lr=0.00257  train_loss=0.6921  train_acc=0.9147  val_acc=0.9710  val_acc_em